# Capstone: forecast_networks

Fairness check for the load forecast: gradient boosting with the home's own information beat
persistence and her networks. Here her GRU, CNN-GRU and BiLSTM (RNN2) get the same chances: the
absolute-error loss as well as hers, and the habit forecasts and appliance states as extra hourly
channels. REFIT (20 homes) then UK-DALE (5 homes), same test hours as before.

**Settings:** Accelerator = **GPU T4**, Internet = **On** (London weather from PVGIS), dataset
`capstone-bundle` (latest version) under Input. Save Version > Save & Run All (Commit).

**A new notebook copies none of these settings.**

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')
sys.path.insert(0, 'scripts')
from kaggle_steps import ADDRESS, notify, run_step
print('ntfy:', 'on' if ADDRESS else 'off (no NTFY_URL secret or first-line topic)')
notify('forecast_networks: started', 'REFIT then UK-DALE', 'default', 'rocket')

In [ ]:
ok = run_step('Tests', 'python -m pytest -q tests/test_energy.py', 'logs/tests.log', summary_lines=2)
ok &= run_step('REFIT networks, same chances', 'python -u scripts/eval_forecast_habits.py --deep',
               'logs/refit_forecast_networks.log', summary_lines=30)
ok &= run_step('UK-DALE networks, same chances', 'python -u scripts/eval_forecast_habits.py --deep',
               'logs/ukdale_forecast_networks.log', env={'CAPSTONE_DATASET': 'ukdale'}, summary_lines=30)

In [ ]:
import zipfile, pathlib
with zipfile.ZipFile('/kaggle/working/forecast_networks.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in [*pathlib.Path('data/results').rglob('forecast_habits_deep.csv'), *pathlib.Path('logs').rglob('*.log')]:
        zf.write(f, str(f))
print(round(pathlib.Path('/kaggle/working/forecast_networks.zip').stat().st_size / 1e6, 2), 'MB')
shutil.rmtree('/kaggle/working/capstone', ignore_errors=True)  # keep the Output to the zip
notify('forecast_networks: FINISHED' if ok else 'forecast_networks: finished WITH ERRORS',
       'forecast_networks.zip is in the Output tab', 'default' if ok else 'high', 'tada' if ok else 'warning')